# S11 toy — budgets, routing, and the privacy boundary

A **personal-finance report generator** ("ledger bot"): three months of card transactions in,
a monthly spending report out. The pipeline has three phases — `categorize`, `summarize`,
`format` — and this session is about *where each phase runs* and *when the run stops*.

Everything below is standard library only. Every "model" is a plain Python function returning
an API-shaped dict; latency and cost are simulated numbers, not sleeps or invoices. Run
top-to-bottom, and write your predictions down before each experiment — a prediction you
didn't write down is a prediction you'll retroactively fix.

## The fixtures

Three months of transactions, with ground-truth categories. Eyeball them first: one month is
built to be tricky for a weak summarizer — two categories finish close together.

In [ ]:
import json
import statistics

CATEGORIES = ["groceries", "dining", "transport", "fun"]

# (date, merchant, amount, true_category)
FIXTURES = {
    "2026-03": [
        ("Mar 02", "CITYGROCER #114",  74.10, "groceries"),
        ("Mar 05", "NOODLE HOUSE",     18.60, "dining"),
        ("Mar 07", "METRO PASS",       21.80, "transport"),
        ("Mar 09", "CITYGROCER #114",  66.30, "groceries"),
        ("Mar 14", "CINEMA REX",       12.00, "fun"),
        ("Mar 21", "GREENGROCER",      22.40, "groceries"),
    ],
    "2026-04": [
        ("Apr 02", "CITYGROCER #114",  45.20, "groceries"),
        ("Apr 04", "STEAKHOUSE 22",    88.40, "dining"),
        ("Apr 06", "METRO PASS",       21.80, "transport"),
        ("Apr 11", "FARMERS MARKET",   38.90, "groceries"),
        ("Apr 18", "SUSHI BAR",        54.40, "dining"),
        ("Apr 19", "GREENGROCER",      46.40, "groceries"),
        ("Apr 25", "CINEMA REX",       12.00, "fun"),
    ],
    "2026-05": [
        ("May 03", "CITYGROCER #114",  58.20, "groceries"),
        ("May 05", "METRO PASS",       21.80, "transport"),
        ("May 09", "BIKE SHOP — SERVICE", 140.00, "transport"),
        ("May 12", "THAI PALACE",      33.50, "dining"),
        ("May 20", "CINEMA REX",       24.00, "fun"),
        ("May 27", "CITYGROCER #114",  49.90, "groceries"),
    ],
}

def totals(month):
    """Deterministic ground truth: category totals (integer cents, no float drift)."""
    cents = {c: 0 for c in CATEGORIES}
    for _, _, amount, cat in FIXTURES[month]:
        cents[cat] += round(amount * 100)
    return {k: v / 100 for k, v in cents.items()}

for m in FIXTURES:
    t = totals(m)
    top = max(t, key=t.get)
    print(m, {k: f"{v:.2f}" for k, v in t.items()},
          f"| grand total {sum(t.values()):.2f} | top: {top}")

## The routes — three mock endpoints

Same API shape as a chat-completions response, plus simulated latency. The routes differ in
price, speed, and *skill at summarizing*:

| route | location | $/1k tokens | latency model | summarize skill |
|---|---|---|---|---|
| `local-small` | local | 0.01 | 30 ms + 0.15 ms/token | weak: close category race → names the wrong winner |
| `local-large` | local | 0.06 | 140 ms + 0.75 ms/token | strong |
| `cloud-frontier` | cloud | 0.30 | 260 ms + 0.12 ms/token | strong |

All three get `categorize` and `format` right — those phases are mechanical. The skill gap
lives entirely in `summarize`, which is the point: difficulty is a *per-phase* property.

In [ ]:
def _resp(content, prompt_tokens, completion_tokens, latency_ms):
    """API-shaped response body, plus timing metadata."""
    return {
        "choices": [{"message": {"role": "assistant", "content": content}}],
        "usage": {"prompt_tokens": prompt_tokens,
                  "completion_tokens": completion_tokens,
                  "total_tokens": prompt_tokens + completion_tokens},
        "latency_ms": latency_ms,
    }

def _categorize_usage(txns):  return 180 + 22 * len(txns), 6 * len(txns)
def _summarize_usage(totals_): return 420 + 18 * len(totals_), 160
def _format_usage(totals_):    return 140 + 10 * len(totals_), 12 * len(totals_)

KEYWORDS = {"CITYGROCER": "groceries", "GREENGROCER": "groceries",
            "FARMERS MARKET": "groceries", "NOODLE": "dining",
            "STEAKHOUSE": "dining", "SUSHI": "dining", "THAI": "dining",
            "METRO": "transport", "BIKE": "transport", "CINEMA": "fun"}

def _categorize(txns):
    def classify(merchant):
        for kw, cat in KEYWORDS.items():
            if kw in merchant:
                return cat
        return "groceries"  # fallback guess
    return [[merchant, classify(merchant)] for _, merchant, _, _ in txns]

def _summarize(totals_, skill):
    ranked = sorted(totals_.items(), key=lambda kv: (-kv[1], kv[0]))
    (top_cat, top_val), (runner_cat, runner_val) = ranked[0], ranked[1]
    claimed = top_cat
    if skill == "weak" and (top_val - runner_val) < 0.10 * runner_val:
        claimed = runner_cat  # the weak model's slop: a close race → wrong winner
    total = sum(totals_.values())
    return (f"FINAL: Total spend {total:.2f}. Top category: {claimed} "
            f"({totals_[claimed]:.2f}). Others: "
            + ", ".join(f"{k} {v:.2f}" for k, v in ranked[1:]))

def _format(totals_):
    lines = ["| category | total |", "|---|---|"]
    for k, v in sorted(totals_.items(), key=lambda kv: -kv[1]):
        lines.append(f"| {k} | {v:.2f} |")
    return "\n".join(lines)

ROUTE_PROFILES = {
    "local-small":    {"location": "local", "usd_per_1k": 0.01,
                       "base_ms": 30,  "ms_per_token": 0.15},
    "local-large":    {"location": "local", "usd_per_1k": 0.06,
                       "base_ms": 140, "ms_per_token": 0.75},
    "cloud-frontier": {"location": "cloud", "usd_per_1k": 0.30,
                       "base_ms": 260, "ms_per_token": 0.12},
}

def make_model(profile, summarize_skill):
    """A mock endpoint: deterministic behavior + API-shaped response + simulated latency."""
    def model(phase, payload):
        if phase == "categorize":
            p, c = _categorize_usage(payload)
            content = json.dumps(_categorize(payload))
        elif phase == "summarize":
            p, c = _summarize_usage(payload)
            content = _summarize(payload, skill=summarize_skill)
        else:  # format
            p, c = _format_usage(payload)
            content = _format(payload)
        return _resp(content, p, c,
                     profile["base_ms"] + profile["ms_per_token"] * (p + c))
    return model

ROUTES = {
    "local-small":    {**ROUTE_PROFILES["local-small"],
                       "model": make_model(ROUTE_PROFILES["local-small"], "weak")},
    "local-large":    {**ROUTE_PROFILES["local-large"],
                       "model": make_model(ROUTE_PROFILES["local-large"], "strong")},
    "cloud-frontier": {**ROUTE_PROFILES["cloud-frontier"],
                       "model": make_model(ROUTE_PROFILES["cloud-frontier"], "strong")},
}

## The privacy boundary, as code

Every phase carries a **data classification**; every route carries a **location**. The
invariant: content-classified phases resolve to local routes. `validate_policy` runs *before*
any model call and **refuses** — it raises, it does not warn. A warning is a log line nobody
reads during the incident; a refusal makes the misconfiguration un-runnable.

In [ ]:
class PolicyRefusal(Exception):
    """Raised at validation, before any model call: content routed off-local."""

PHASES = {
    "categorize": "content",   # raw transaction descriptions
    "summarize":  "content",   # prose about your spending
    "format":     "metadata",  # category totals only, no descriptions
}

def validate_policy(route_table):
    """The boundary as a validation step. Refuses to run; never warns."""
    for phase, classification in PHASES.items():
        route = route_table.get(phase)
        if route not in ROUTES:
            raise PolicyRefusal(f"{phase}: unknown route {route!r}")
        if classification == "content" and ROUTES[route]["location"] != "local":
            raise PolicyRefusal(
                f"{phase} handles {classification} data but routes to {route} "
                f"({ROUTES[route]['location']}) — content phases stay local")

## The metered pipeline

`run_pipeline` validates the route table, runs the phases, and meters every call
(tokens × route price, latency). Crossing `budget_usd` ends the run with
`stop_reason="budget_exceeded"` — a first-class outcome, not a crash. The summarize phase
may take several passes: a well-behaved model returns `FINAL:` on the first; a refinement
loop with no convergence criterion never does, and then only the budget (or the pass cap)
stands between you and the invoice.

In [ ]:
def run_pipeline(month, route_table, budget_usd=None, max_passes=1, routes=None):
    routes = ROUTES if routes is None else routes
    validate_policy(route_table)                 # refuses BEFORE any call is made
    txns, t = FIXTURES[month], totals(month)
    log = {"calls": [], "cost_usd": 0.0, "outputs": {},
           "passes": 0, "stop_reason": "ok"}

    def over_budget():
        return budget_usd is not None and log["cost_usd"] > budget_usd

    def call(phase, payload):
        r = routes[route_table[phase]]
        resp = r["model"](phase, payload)
        tok = resp["usage"]["total_tokens"]
        log["cost_usd"] += tok / 1000 * r["usd_per_1k"]
        log["calls"].append({"phase": phase, "route": route_table[phase],
                             "tokens": tok, "cost_usd": log["cost_usd"],
                             "latency_ms": round(resp["latency_ms"], 1)})
        return resp["choices"][0]["message"]["content"]

    log["outputs"]["categorize"] = json.loads(call("categorize", txns))
    if over_budget():
        log["stop_reason"] = "budget_exceeded"
        return log

    text = ""
    for k in range(1, max_passes + 1):
        log["passes"] = k
        text = call("summarize", t)
        if over_budget():
            log["stop_reason"] = "budget_exceeded"
            return log
        if text.startswith("FINAL:"):
            break
    else:
        log["stop_reason"] = "pass_cap"          # never converged within the cap
        return log
    log["outputs"]["summarize"] = text

    log["outputs"]["format"] = call("format", t)
    if over_budget():
        log["stop_reason"] = "budget_exceeded"
    return log

def check_month(month, outputs):
    """Deterministic tier: structure and ground truth, no vibes."""
    t = totals(month)
    top = max(t, key=t.get)
    total_str = f"{sum(t.values()):.2f}"
    got = {m: c for m, c in outputs["categorize"]}
    truth = {m: c for _, m, _, c in FIXTURES[month]}
    text = outputs["summarize"].lower()
    return {
        "categorize": got == truth,
        "summarize": f"top category: {top}" in text and total_str in text,
        "format": all(f"| {k} | {v:.2f} |" in outputs["format"]
                      for k, v in t.items()),
    }

def run_suite(route_table, budget_usd=None):
    rows = []
    for month in FIXTURES:
        log = run_pipeline(month, route_table, budget_usd=budget_usd)
        checks = (check_month(month, log["outputs"])
                  if log["stop_reason"] == "ok" else {})
        rows.append({"month": month, "log": log, "checks": checks,
                     "pass": bool(checks) and all(checks.values())})
    return rows

def show_suite(name, rows):
    passed = sum(r["pass"] for r in rows)
    cost = sum(r["log"]["cost_usd"] for r in rows)
    print(f"{name:34s} pass {passed}/{len(rows)}   cost ${cost:.4f}")
    for r in rows:
        failed = [k for k, v in r["checks"].items() if not v] or ["-"]
        print(f"    {r['month']}  {'PASS' if r['pass'] else 'FAIL'}"
              f"  ${r['log']['cost_usd']:.4f}  failed: {', '.join(failed)}")
    return rows

ALL_SMALL = {"categorize": "local-small", "summarize": "local-small",
             "format": "local-small"}
ALL_LARGE = {"categorize": "local-large", "summarize": "local-large",
             "format": "local-large"}

## Experiment 1 — the meandering run

`make_chatty_model` wraps `local-large`'s summarize in a refinement loop with **no
convergence criterion**: every draft "needs one more polish." The engine gives it
`max_passes=12`.

**Predict first:** (a) with no budget, what `stop_reason` ends the run and what does it
cost? (b) with `budget_usd=0.25`, after *exactly which* summarize pass does the run die,
and with what `stop_reason`? Write both down before running the solution cell.

In [ ]:
# YOUR PREDICTION (fill in before running the solution cell):
predicted_no_budget_cost = None   # e.g. 0.49
predicted_stop_pass = None        # the pass number the budgeted run dies on
predicted_stop_reason = None      # "budget_exceeded" / "pass_cap" / "ok"

def make_chatty_model(profile):
    """Strong summarize that never converges: every draft needs one more pass."""
    base = make_model(profile, "strong")
    def model(phase, payload):
        if phase == "summarize":
            p, c = _summarize_usage(payload)
            return _resp("DRAFT: almost there — one more polish…", p, c,
                         profile["base_ms"] + profile["ms_per_token"] * (p + c))
        return base(phase, payload)
    return model

CHATTY_ROUTES = {**ROUTES, "local-large":
                 {**ROUTES["local-large"],
                  "model": make_chatty_model(ROUTE_PROFILES["local-large"])}}

# (a) no budget at all — watch it meander to the cap:
log = run_pipeline("2026-03", ALL_LARGE, max_passes=12, routes=CHATTY_ROUTES)
print("stop_reason:", log["stop_reason"], "| passes:", log["passes"],
      f"| cost ${log['cost_usd']:.4f}")

In [ ]:
# SOLUTION — run only after your prediction is written down.
# (b) same chatty model, now the harness carries a $0.25 run budget:
log = run_pipeline("2026-03", ALL_LARGE, budget_usd=0.25,
                   max_passes=12, routes=CHATTY_ROUTES)
for i, c in enumerate(log["calls"], 1):
    print(f"  call {i}: {c['phase']:10s} cumulative ${c['cost_usd']:.4f}")
print("stop_reason:", log["stop_reason"], "| died after pass:", log["passes"],
      f"| spent ${log['cost_usd']:.4f} of $0.25")
print("your prediction:", predicted_stop_pass, predicted_stop_reason)
assert log["stop_reason"] == "budget_exceeded"
# A run that costs more than the task is worth has failed — even mid-'progress'.

## Experiment 2 — measure the tiers

Same fixtures, same deterministic checker, same everything except the route table — the
diverge/rejoin discipline from S02.

**Predict first:** for each phase, does `local-small` hold its number? And the suite pass
counts: `all-small` ?/3, `all-large` ?/3. (Remember which month was built tricky.)

In [ ]:
# YOUR PREDICTION (fill in before running the solution cell):
predicted_small_holds = {"categorize": None, "summarize": None, "format": None}
predicted_suite_pass = {"all-small": None, "all-large": None}   # out of 3

In [ ]:
# SOLUTION
rows_small = show_suite("all-small", run_suite(ALL_SMALL))
print()
rows_large = show_suite("all-large", run_suite(ALL_LARGE))
print()
print("your phase predictions:", predicted_small_holds)
# all-small is ~6x cheaper and fails exactly where the skill gap lives: summarize,
# on the month where the top category race is close.

## Experiment 3 — the cheapest table that holds the number

Now the actual engineering move: pick a route per phase targeting **suite parity with
all-large at minimum cost**. The constraint is the number, not the answer key — any table
that passes 3/3 cheaper than the solution is a better answer.

**Predict first:** write your table, predict its cost, then run.

In [ ]:
# YOUR ROUTE TABLE — cheapest config that still passes 3/3.
# Routes: "local-small", "local-large", "cloud-frontier" (mind the boundary!)
my_route_table = {
    "categorize": "local-large",   # TODO: your choice
    "summarize":  "local-large",   # TODO: your choice
    "format":     "local-large",   # TODO: your choice
}
# show_suite("mine", run_suite(my_route_table))   # uncomment to test yours

In [ ]:
# SOLUTION — one cheapest table that holds the number:
routed = {"categorize": "local-small",
          "summarize":  "local-large",
          "format":     "local-small"}
rows_routed = show_suite("routed (small cat/fmt, large sum)", run_suite(routed))
cost_routed = sum(r["log"]["cost_usd"] for r in rows_routed)
cost_large = sum(r["log"]["cost_usd"] for r in run_suite(ALL_LARGE))
print(f"\nsavings vs all-large: {100 * (1 - cost_routed / cost_large):.1f}% "
      f"at equal pass rate")
assert all(r["pass"] for r in rows_routed) and cost_routed < cost_large
# The deliverable is not this table — it is the measurement the table cites.

## Experiment 4 — the privacy refusal

Someone "optimizes" the table and points `summarize` at `cloud-frontier` — strong *and*
faster than local-large. Tempting.

**Predict first:** when does this fail — at validation or mid-run — and *how many model
calls happen before it fails*? Then the subtle case: `format` carries metadata only. Does
the boundary allow it on cloud, and does the bill agree?

In [ ]:
# YOUR PREDICTION (fill in before running the solution cell):
predicted_calls_before_refusal = None   # how many model calls before it dies?
predicted_format_on_cloud = None        # "allowed" / "refused"

misconfigured = {"categorize": "local-small",
                 "summarize":  "cloud-frontier",   # content phase → cloud route
                 "format":     "local-small"}
# try:
#     run_pipeline("2026-03", misconfigured)
# except PolicyRefusal as e:
#     print("REFUSED:", e)

In [ ]:
# SOLUTION
try:
    run_pipeline("2026-03", misconfigured)
    raise AssertionError("should have refused")
except PolicyRefusal as e:
    print("REFUSED, before any model call:", e)
# validate_policy is run_pipeline's first statement — zero calls happen first.

# The subtle case: format is metadata-only, so cloud IS allowed there...
cloud_format = {"categorize": "local-small",
                "summarize":  "local-large",
                "format":     "cloud-frontier"}
rows_cf = show_suite("routed + cloud format", run_suite(cloud_format))
cost_cf = sum(r["log"]["cost_usd"] for r in rows_cf)
print(f"\ncloud format costs ${cost_cf:.4f} vs ${cost_routed:.4f} all-local routed")
# ...and it is strictly worse on the bill. The boundary permits it; arithmetic forbids it.
# Classification decides what is allowed — never vendor vibes. Allowed != wise.

## Experiment 5 — the voice budget

Latency is a budget with a human attached. Human turn-taking clusters around ~250 ms
response offsets; much over ~1 s of dead air reads as broken. Suppose a voice mode where
each user question triggers `categorize` + `summarize` (the content phases) plus 150 ms of
fixed ASR/TTS overhead. Per-turn budget: **1000 ms at p50**.

**Predict first:** which configs fit — all-large, routed? Which phase dominates the loser?
And what would the *forbidden* fix (frontier cloud for content) have cost in latency?

In [ ]:
# YOUR PREDICTION (fill in before running the solution cell):
VOICE_OVERHEAD_MS = 150
VOICE_BUDGET_MS = 1000
predicted_voice = {"all-large": None, "routed": None}   # "fits" / "blows"
predicted_dominant_phase = None

In [ ]:
# SOLUTION
def turn_latencies(route_table):
    """Per-month voice-turn latency: categorize + summarize + fixed overhead."""
    out = []
    for r in run_suite(route_table):
        by_phase = {c["phase"]: c["latency_ms"] for c in r["log"]["calls"]}
        out.append(by_phase["categorize"] + by_phase["summarize"]
                   + VOICE_OVERHEAD_MS)
    return out

p50 = {}
for name, table in [("all-large", ALL_LARGE), ("routed", routed)]:
    ts = turn_latencies(table)
    p50[name] = statistics.median(ts)
    verdict = "fits" if p50[name] <= VOICE_BUDGET_MS else "BLOWS"
    print(f"{name:10s} turns {[f'{t:.0f}' for t in ts]} ms"
          f"  p50={p50[name]:.0f} ms -> {verdict} the {VOICE_BUDGET_MS} ms budget")
assert p50["routed"] <= VOICE_BUDGET_MS < p50["all-large"]

# The forbidden fix: content phases on cloud-frontier.
prof = ROUTE_PROFILES["cloud-frontier"]
cloud_turn = (prof["base_ms"] + prof["ms_per_token"] * 348      # categorize
              + prof["base_ms"] + prof["ms_per_token"] * 652    # summarize
              + VOICE_OVERHEAD_MS)
print(f"\nfrontier-cloud turn would be ~{cloud_turn:.0f} ms — fastest, and off-limits:"
      f" content stays local. Latency gets engineered (streaming, precompute),"
      f" not bought. That is why this math happens before the build, not after.")

## What transfers

- **Budgets are runtime invariants.** A meter in the loop, a named `stop_reason`, a limit
  set from the task's value — not a finance afterthought discovered on the invoice.
- **Route tables are policy-as-data.** Diffable, reviewable, validatable without touching
  the engine — and the shipped table cites a suite number: the cheapest route that *holds
  the number*, measured under diverge/rejoin.
- **The privacy boundary refuses, it doesn't warn.** Validation before any call; data
  classification decides, not vendor virtue; "allowed" and "wise" are different questions
  (cloud `format` was both legal and more expensive).
- **Latency is a budget with a human attached.** p50 per turn against the ~250 ms
  turn-taking base rate and a ~1 s dead-air ceiling; the boundary removes the easy escape,
  so the math happens on paper first.
- **Failure modes to recognize in the wild:** warnings where refusals belong; route choices
  that cite no measurement; mean-latency quotes that hide the p95 users actually remember.